# 00 — Create 2024–2025 Balanced Sample

I start from the original Kaggle export: https://www.kaggle.com/datasets/asaniczka/trending-youtube-videos-113-countries/data

- `trending_yt_videos_113_countries.csv`

I create:

- `youtube_2024_2025_balanced_sample.csv`

These are the steps:
1. read the large CSV in chunks
2. keep only rows from 2024 and 2025
3. combine them
4. take up to 100,000 rows per year
5. save the final balanced sample

A fixed seed of `42` is used so the result is reproducible.


In [1]:
import pandas as pd
from pathlib import Path

SEED = 42
CHUNKSIZE = 500_000

BASE_DIR = Path(".")
RAW_FILE = BASE_DIR / "trending_yt_videos_113_countries.csv"
OUTPUT_FILE = BASE_DIR / "youtube_2024_2025_balanced_sample.csv"

RAW_FILE


WindowsPath('trending_yt_videos_113_countries.csv')

In [2]:
filtered_chunks = []

for chunk in pd.read_csv(RAW_FILE, chunksize=CHUNKSIZE, low_memory=False):
    chunk["snapshot_date"] = pd.to_datetime(chunk["snapshot_date"], errors="coerce")
    chunk["year"] = chunk["snapshot_date"].dt.year
    chunk["month"] = chunk["snapshot_date"].dt.month

    chunk = chunk[chunk["year"].isin([2024, 2025])].copy()

    if not chunk.empty:
        filtered_chunks.append(chunk)

df_24_25 = pd.concat(filtered_chunks, ignore_index=True)

print("Filtered shape:", df_24_25.shape)
print(df_24_25["year"].value_counts().sort_index())


Filtered shape: (4096817, 20)
year
2024    2053919
2025    2042898
Name: count, dtype: int64


In [3]:
final_parts = []

for year, grp in df_24_25.groupby("year"):
    grp = grp.copy()
    if len(grp) > 100_000:
        grp = grp.sample(n=100_000, random_state=SEED)
    final_parts.append(grp)

df_final = pd.concat(final_parts, ignore_index=True)
df_final = df_final.sample(frac=1, random_state=SEED).reset_index(drop=True)

df_final.to_csv(OUTPUT_FILE, index=False)

print("Saved:", OUTPUT_FILE)
print("Final shape:", df_final.shape)
print("\nRows by year:")
print(df_final["year"].value_counts().sort_index())
print("\nMonth distribution by year:")
display(df_final.groupby(["year", "month"]).size().unstack(fill_value=0))


Saved: youtube_2024_2025_balanced_sample.csv
Final shape: (200000, 20)

Rows by year:
year
2024    100000
2025    100000
Name: count, dtype: int64

Month distribution by year:


month,1,2,3,4,5,6,7,8,9,10,11,12
year,,,,,,,,,,,,
2024,8560,7969,8143,8311,8537,8203,8321,8618,8238,8354,8289,8457
2025,8416,7776,8555,8326,8562,8402,8570,8449,8083,8395,8097,8369
